In [104]:
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel
load_dotenv()


client=Anthropic()
model='claude-sonnet-4-5'

In [105]:
from anthropic.types import Message


system=system = """
You have three tools available: read_file, write_file, and web_search.
- Use web_search ONLY for searching the internet
- Use write_file ONLY for saving files — NEVER use code_execution to write files
- NEVER write files to /tmp/ or any server path
- ALL files must be saved using the write_file tool directly
- Do NOT use code_execution for anything other than what is absolutely necessary
"""

In [ ]:
def add_user_message(conversation,user_message):
    user_message={"role":"user","content":user_message.content if isinstance(user_message,Message) else user_message}
    conversation.append(user_message)
    

def add_assistant_message(conversation,assistant_message):
    assistant_message={"role":"assistant","content":assistant_message.content if isinstance(assistant_message,Message) else assistant_message}
    conversation.append(assistant_message)

def call_llm(
        messages,
        max_tokens=1200,
        tools=None,
        streaming=False,
        system=system,
        model=model,container=None):
    params={"messages":messages,
        "model":model,
        "max_tokens":max_tokens,
        "model":model,"system":system
        # "tool_choice":{"type": "any"}
    }
    if container:
        params["container"]=container
    if tools:
        params["tools"]=tools
    if streaming:
        params["streaming"]=True,
    response=client.messages.create(**params)
    return response

    

In [ ]:
#Tool for the websearch
websearch_tool={
    "type":"web_search_20260209",
    "name":"web_search",
    "max_uses":3,
}

#Tool to read file
def read_file(file_name):
    with open(file_name,'+r') as f:
            file_content=f.read()
    return f"read_file_status: FILE READ SUCCESSFUL here are the file read contents : {file_content}"
    
    
read_file_schema={
        "name": "read_file",
        "description": "Reads the full text content of a local file from disk and returns it as a string. Use this when you need to access information stored in a local .txt or .md file. Only use this for files you know exist on disk. Returns the raw string content of the file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_name": {
                    "type": "string",
                    "description": "The name or relative path of the file to read. For example: 'notes.txt' or 'data/report.md'"
                }
            },
            "required": ["file_name"]
        }
    }

#Tool for writing in the file
def write_file(file_name,content):
    with open(file_name,'w',encoding='utf-8') as f:
            f.write(content)
    return "write_file_status:FILE WRITE WAS SUCCESSFUL"
    

write_file_schema={
        "name": "write_file",
        "description": "Writes the given text content to a local file on disk. Use this to save reports, summaries, or any text output. If the file already exists, it will be completely overwritten. Use this as the final step when you are ready to produce the output report.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_name": {
                    "type": "string",
                    "description": "The name or relative path of the file to write to. For example: 'report.md' or 'output/summary.txt'"
                },
                "content": {
                    "type": "string",
                    "description": "The full text content to write into the file."
                }
            },
            "required": ["file_name", "content"]
        }
    }


In [108]:
tool_lookup={
    "read_file":read_file,
    "write_file":write_file
}

In [109]:
from anthropic.types import ToolUseBlock
from anthropic.types import TextBlock

In [110]:

def tool_executor(tool_block):
    try:
        tool=tool_lookup[tool_block.name]
        tool_response=tool(**tool_block.input)
        return {
            "type":"tool_result",
            "tool_use_id":tool_block.id,
            "content":tool_response,
            "is_error":False
        }
    except Exception as e:
        return {
            "type":"tool_result",
            "tool_use_id":tool_block.id,
            "content":f"Error{e}",
            "is_error":True
        }

In [111]:
container_id=None
agent_loop_messages=[]
user_message=input()
print(user_message)
add_user_message(agent_loop_messages,user_message)
while True:
    response=call_llm(agent_loop_messages,tools=[read_file_schema,write_file_schema,websearch_tool],container=container_id)
    add_assistant_message(agent_loop_messages,response)
    if response.container:
        container_id=response.container.id
    print(response)
    if(response.stop_reason=='max_tokens'):
        print("oh my gawd the tokens are finished I am leaving")
        break
    if(response.stop_reason=='end_turn'):
        for block in response.content:
            if isinstance(block,TextBlock):
                print(block.text)
        break
    if(response.stop_reason=='tool_use'):
        tool_responses=[]
        for block in response.content:
            if isinstance(block,TextBlock):
                print(block.text)
            if isinstance(block,ToolUseBlock):
                tool_executor_response=tool_executor(block)
                tool_responses.append(tool_executor_response)
        add_user_message(agent_loop_messages,tool_responses)
print("okay bye bye the task has been completed")   

       

Do one thing, I want you to think about five unique research ideas(topics only). Write it in a file named as ResearchTopics.txt. in my local system .Then i want you to go with each of the topic and search the web for it and then make a file and write the findings each in an individual file name called idea_topic.txt
Message(id='msg_01TeJcmeMyXecfa7RxAzeXt5', container=None, content=[TextBlock(citations=None, text="I'll help you create five unique research ideas and then search the web for each topic to document the findings. Let me start by creating the research topics file.", type='text'), ToolUseBlock(id='toolu_01HqEboC7vKW1cMeADs77P1M', caller=DirectCaller(type='direct'), input={'file_name': 'ResearchTopics.txt', 'content': 'Five Unique Research Ideas\n===========================\n\n1. Neuroplasticity in Adult Learning Through Virtual Reality Environments\n   - Investigating how immersive VR experiences can enhance brain plasticity and accelerate skill acquisition in adults\n\n2. Mi

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 24fc1258-fd1b-4718-945a-b2e1b290f5b5, model: claude-sonnet-4-5-20250929). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CaLNXavogmna3vSgj7GCA'}

In [ ]:
agent_loop_messages

[{'role': 'user',
  'content': 'Do one thing, I want you to think about five unique research ideas(topics only). Write it in a file named as ResearchTopics.txt. in my local system .Then i want you to go with each of the topic and search the web for it and then make a file and write the findings each in an individual file name called idea_topic.txt'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you with that! Let me break this down into steps:\n\n1. First, I'll create 5 unique research topics and save them to ResearchTopics.txt\n2. Then I'll search the web for each topic\n3. Finally, I'll create individual files with findings for each topic\n\nLet's start:", type='text'),
   ToolUseBlock(id='toolu_01VPnGRsnirUEffDYrzR6EzG', caller=DirectCaller(type='direct'), input={'file_name': 'ResearchTopics.txt', 'content': 'Five Unique Research Ideas:\n\n1. Quantum Machine Learning for Drug Discovery\n   - Exploring the intersection of quantum computing and AI to 

In [ ]:
response

Message(id='msg_014zgmZcP3zKodX6g2Z6CJNH', container=None, content=[ToolUseBlock(id='toolu_01NeXKf35y4LCL5yWusKPaMf', caller=DirectCaller(type='direct'), input={'file_name': 'idea_vertical_farming_closed_loop_nutrient_recycling.txt'}, name='write_file', type='tool_use')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9764, output_tokens=1200, server_tool_use=None, service_tier='standard'))

In [ ]:
agent_loop_messages[9]

{'role': 'assistant',
 'content': [TextBlock(citations=None, text="Now let me create files for the remaining two topics. Since I hit the search limit, I'll create comprehensive content based on current knowledge:", type='text'),
  ToolUseBlock(id='toolu_012X2eU2qi1u3cwDwS2ahpEn', caller=DirectCaller(type='direct'), input={'file_name': 'idea_topic4.txt'}, name='write_file', type='tool_use')]}